In [3]:
import numpy as np
from collections import Counter


# ---------------------------------------------------------
# 1. Gini Impurity
# ---------------------------------------------------------

def gini(y):
    counts = Counter(y)
    n = len(y)

    return 1 - sum((count / n) ** 2 for count in counts.values())


# ---------------------------------------------------------
# 2. Split Dataset
# ---------------------------------------------------------

def split(X, y, feature, threshold):

    left = X[:, feature] <= threshold
    right = X[:, feature] > threshold

    return (
        X[left], y[left],
        X[right], y[right]
    )


# ---------------------------------------------------------
# 3. Find Best Split
# ---------------------------------------------------------

def best_split(X, y):

    best_gain = -1
    best_feature = None
    best_threshold = None

    parent_gini = gini(y)

    for feature in range(X.shape[1]):

        for threshold in np.unique(X[:, feature]):

            X_left, y_left, X_right, y_right = split(
                X, y, feature, threshold
            )

            if len(y_left) == 0 or len(y_right) == 0:
                continue

            n = len(y)

            weighted_gini = (
                len(y_left) / n * gini(y_left)
                +
                len(y_right) / n * gini(y_right)
            )

            gain = parent_gini - weighted_gini

            if gain > best_gain:
                best_gain = gain
                best_feature = feature
                best_threshold = threshold

    return best_feature, best_threshold


# ---------------------------------------------------------
# 4. Decision Tree
# ---------------------------------------------------------

class DecisionTree:

    def __init__(self, max_depth=3):
        self.max_depth = max_depth
        self.tree = None

    def fit(self, X, y):
        self.tree = self.build_tree(X, y, depth=0)

    def build_tree(self, X, y, depth):

        # Stop if all samples belong to same class
        if len(set(y)) == 1:
            return Counter(y).most_common(1)[0][0]

        # Stop if maximum depth reached
        if depth == self.max_depth:
            return Counter(y).most_common(1)[0][0]

        feature, threshold = best_split(X, y)

        # No valid split
        if feature is None:
            return Counter(y).most_common(1)[0][0]

        X_left, y_left, X_right, y_right = split(
            X, y, feature, threshold
        )

        return {
            "feature": feature,
            "threshold": threshold,
            "left": self.build_tree(
                X_left, y_left, depth + 1
            ),
            "right": self.build_tree(
                X_right, y_right, depth + 1
            )
        }

    def predict_one(self, x, node):

        # Leaf node
        if not isinstance(node, dict):
            return node

        if x[node["feature"]] <= node["threshold"]:
            return self.predict_one(x, node["left"])

        return self.predict_one(x, node["right"])

    def predict(self, X):

        return np.array([
            self.predict_one(x, self.tree)
            for x in X
        ])

In [4]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# creae random dataset
X, y = make_classification(
    n_samples=200,
    n_features=2,
    n_informative=2,
    n_redundant=0,
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Decision Tree

# 1. Calculate Gini
#         ↓
# 2. Try feature + threshold
#         ↓
# 3. Calculate Gini after split
#         ↓
# 4. Choose highest gain
#         ↓
# 5. Recursively split
#         ↓
# 6. Stop at max depth / pure node
#         ↓
# 7. Leaf predicts majority class

tree = DecisionTree(max_depth=3)

tree.fit(X_train, y_train)

pred = tree.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))

Accuracy: 0.8


In [5]:
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(
    criterion="gini",
    max_depth=3,
    min_samples_split=2,
    min_samples_leaf=1
)

model.fit(X_train, y_train)

pred = model.predict(X_test)